In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "REPLACE_AFTER_PUSH"
assert (len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"), "Pin the reviewed pushed diagnostic implementation commit before Run all"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
DIAGNOSTIC_VERSION = "v1_frozen_coca_df_embeddings"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
CLASSIFIER_ROOT = SHARED_RUN_ROOT / "sqrt_balanced_seed0_v1" / "coca_classifier"
V4_ROOT = CLASSIFIER_ROOT / "v4_focal_inverse_frequency"
V4_FAILURE_RECORD = V4_ROOT / "latest_validation_failure.json"
DIAGNOSTIC_ROOT = CLASSIFIER_ROOT / "embedding_diagnostics" / DIAGNOSTIC_VERSION
LATEST_RECORD = DIAGNOSTIC_ROOT / "latest_diagnostic_record.json"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".coca_shared_root.json"
CANDIDATE_MANIFEST = SHARED_PROJECT_DIR / "outputs" / "exploratory_balanced_ddpm" / "sqrt_balanced_seed0_v1" / "candidate_synthetic_df" / "epoch0100_seed0" / "synthetic_df.csv"
EXPECTED_CANDIDATE_SHA256 = "9ef9b44e404f74aab8211f4e7d123da3258ba8ba4e3004a4147d1761ed343b34"
EXPECTED_GROUP_COUNTS = {"real_train_df": 85, "synthetic_df": 500, "validation_df": 14}


# Frozen CoCa df embedding diagnostic

Descriptive post-v4-failure diagnostic only. It compares original train-df, synthetic-df, and validation-df embeddings using centroid cosine distance, distance to each group's own centroid, and directional cross-group nearest-neighbour cosine distance. It does not train a classifier, access test data, select a model, or authorize formal runs.

## Phase 0 CHECK — pinned code, shared roots, and immutable v4 failure

In [ ]:
import base64, hashlib, json, os, subprocess, sys
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
assert SHARED_ROOT_SENTINEL.is_file(), "existing shared-root sentinel is required"
assert V4_FAILURE_RECORD.is_file(), "the immutable v4 failure record is required"
assert V4_ROOT not in DIAGNOSTIC_ROOT.parents and DIAGNOSTIC_ROOT != V4_ROOT
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
v4_failure = json.loads(V4_FAILURE_RECORD.read_text(encoding="utf-8"))
assert v4_failure["validation_status"] == "VALIDATION FAILED"
assert v4_failure["formal_training_started"] is False
assert v4_failure["run_version"] == "v4_focal_inverse_frequency"
assert v4_failure["non_collapse_gate"]["C4"]["best_validation_df_f1"] == 0.0
assert v4_failure["non_collapse_gate"]["C4"]["prediction_counts"]["df"] == 0
guard_before = {str(V4_FAILURE_RECORD): (sha256(V4_FAILURE_RECORD), V4_FAILURE_RECORD.stat().st_mtime_ns)}
subprocess.run(["nvidia-smi"], check=True)
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access is required"
CODE_DIR = Path("/content/ddpm-coca-embedding-diagnostic-code")
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status
assert "@" not in remote and "x-access-token" not in remote
os.environ["HF_HOME"] = "/content/hf-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "open_clip_torch==3.3.0", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
from ddpm_derm import coca_run
assert torch.cuda.is_available() and version("open_clip_torch") == "3.3.0"
resolved_root = coca_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = coca_run.probe_shared_drive(resolved_root)
sentinel = json.loads(SHARED_ROOT_SENTINEL.read_text(encoding="utf-8"))
assert sentinel["shortcut_alias"] == "ddpm-derm-coca-runs"
assert sentinel["resolved_path"] == str(resolved_root)


## Phase 1 CHECK — train/validation-only groups and candidate identity

In [ ]:
from datetime import datetime, timezone
assert CANDIDATE_MANIFEST.is_file(), f"candidate manifest missing: {CANDIDATE_MANIFEST}"
candidate_hash = sha256(CANDIDATE_MANIFEST)
assert candidate_hash == EXPECTED_CANDIDATE_SHA256
os.environ["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
from ddpm_derm import coca_embedding_diagnostic
groups = coca_embedding_diagnostic.build_df_groups(CANDIDATE_MANIFEST)
assert {name: len(frame) for name, frame in groups.items()} == EXPECTED_GROUP_COUNTS
assert not LATEST_RECORD.exists(), f"completed diagnostic already exists; do not overwrite: {LATEST_RECORD}"
if DIAGNOSTIC_ROOT.exists():
    prior_attempts = [path for path in DIAGNOSTIC_ROOT.iterdir() if path.is_dir()]
    assert not prior_attempts, f"incomplete diagnostic attempt requires review: {prior_attempts}"
else:
    coca_run.ensure_tree(SHARED_RUN_ROOT, DIAGNOSTIC_ROOT.relative_to(SHARED_RUN_ROOT))
attempt_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
OUTPUT_DIR = coca_run.ensure_tree(SHARED_RUN_ROOT, (DIAGNOSTIC_ROOT / attempt_id).relative_to(SHARED_RUN_ROOT))


## Phase 2 RUN — frozen native-eval embeddings only

In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(CODE_DIR / "src")
env["PYTHONUNBUFFERED"] = "1"
command = [sys.executable, "-u", "-m", "ddpm_derm.coca_embedding_diagnostic", "--generated-manifest", str(CANDIDATE_MANIFEST), "--expected-candidate-sha256", EXPECTED_CANDIDATE_SHA256, "--output-dir", str(OUTPUT_DIR), "--git-commit", commit, "--shared-root-uuid", sentinel["shared_root_uuid"], "--v4-failure-sha256", sha256(V4_FAILURE_RECORD), "--device", "cuda", "--batch-size", "32", "--num-workers", "2"]
process = subprocess.Popen(command, cwd=CODE_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
lines = []
for line in process.stdout:
    lines.append(line)
    print(line, end="", flush=True)
if process.wait():
    raise subprocess.CalledProcessError(process.returncode, command, output="".join(lines))


## Phase 3 REVIEW — reopen artifacts and preserve the evidence boundary

In [ ]:
import numpy as np
record_path = OUTPUT_DIR / "embedding_diagnostic.json"
embedding_path = OUTPUT_DIR / "df_embeddings.npz"
assert record_path.is_file() and embedding_path.is_file()
record = json.loads(record_path.read_text(encoding="utf-8"))
assert record["diagnostic_status"] == "COMPLETED"
assert record["diagnostic_version"] == DIAGNOSTIC_VERSION
assert record["formal_training_started"] is False
assert record["test_data_accessed"] is False
assert record["candidate_manifest_sha256"] == EXPECTED_CANDIDATE_SHA256
assert record["analysis"]["group_counts"] == EXPECTED_GROUP_COUNTS
assert record["analysis"]["feature_dimension"] == 512
assert record["model_identity"]["model_name"] == "coca_ViT-B-32"
assert record["model_identity"]["pretrained_tag"] == "laion2b_s13b_b90k"
assert record["model_identity"]["trainable_parameter_count"] == 3591
with np.load(embedding_path, allow_pickle=False) as saved:
    assert {name: list(saved[name].shape) for name in saved.files} == {"real_train_df": [85, 512], "synthetic_df": [500, 512], "validation_df": [14, 512]}
    for name in saved.files:
        assert np.isfinite(saved[name]).all()
        np.testing.assert_allclose(np.linalg.norm(saved[name], axis=1), 1.0, rtol=1e-5, atol=1e-6)
record["drive_probes"] = drive_probe
record["diagnostic_artifact_directory"] = str(OUTPUT_DIR)
coca_run.write_json_atomic(record_path, record)
coca_run.write_json_atomic(LATEST_RECORD, record)
guard_after = {str(V4_FAILURE_RECORD): (sha256(V4_FAILURE_RECORD), V4_FAILURE_RECORD.stat().st_mtime_ns)}
assert guard_after == guard_before, "v4 failure evidence changed"
print(json.dumps(record["analysis"], indent=2))
print("EMBEDDING DIAGNOSTIC COMPLETED")
print("formal_training_started=false")
print("test_data_accessed=false")
print("Download this executed notebook plus the diagnostic JSON and NPZ for independent review.")
